<a href="https://colab.research.google.com/github/cbadenes/curso-pln/blob/main/notebooks/08_RAG_Basico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG con generación real usando TinyLlama

Este notebook extiende la búsqueda dispersa y densa incorporando un modelo LLM real para generar respuestas.

Usaremos **TinyLlama-1.1B-Chat** para mantenerlo ligero y didáctico.

## 1. Dataset de películas

In [1]:
documents = [
    "Inception es una película de ciencia ficción sobre sueños dentro de sueños.",
    "Interstellar trata sobre viajes espaciales y agujeros negros.",
    "The Dark Knight es una película de superhéroes con el Joker como villano.",
    "La La Land es un musical romántico ambientado en Los Ángeles.",
    "The Matrix explora la realidad simulada y la inteligencia artificial."
]

## 2. Búsqueda dispersa (TF-IDF)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

def sparse_search(query, top_k=2):
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_vectors)[0]
    top_idx = sims.argsort()[::-1][:top_k]
    return [documents[i] for i in top_idx]

## 3. Búsqueda densa (embeddings)

In [3]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(documents)

def dense_search(query, top_k=2):
    query_emb = embed_model.encode([query])[0]
    sims = doc_embeddings @ query_emb
    top_idx = sims.argsort()[::-1][:top_k]
    return [documents[i] for i in top_idx]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. Cargar TinyLlama

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 5. Generador RAG con prompt

In [26]:
def generate_answer(query, context_docs):
    context = "\n".join(context_docs)

    prompt = f"""<|system|>
Eres un experto en cine que proporciona recomendaciones y análisis de películas.
Usa el siguiente contexto para responder la pregunta del usuario.

Contexto:
{context}

<|user|>
{query}

<|assistant|>
Basándome en las películas mencionadas, recomiendo: """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        num_return_sequences=1,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

## 6. Pipeline RAG

In [27]:
def rag_pipeline(query, search_fn):
    docs = search_fn(query)
    answer = generate_answer(query, docs)
    return docs, answer

## 7. Ejemplo

In [28]:
query = "Recomiéndame una película sobre sueños o realidades complejas"

docs_sparse, answer_sparse = rag_pipeline(query, sparse_search)
docs_dense, answer_dense = rag_pipeline(query, dense_search)

print("=== CONTEXTO (Sparse) ===")
print(docs_sparse)
print("\n=== RESPUESTA ===")
print(answer_sparse)

print("\n\n=== CONTEXTO (Dense) ===")
print(docs_dense)
print("\n=== RESPUESTA ===")
print(answer_dense)

=== CONTEXTO (Sparse) ===
['Inception es una película de ciencia ficción sobre sueños dentro de sueños.', 'The Dark Knight es una película de superhéroes con el Joker como villano.']

=== RESPUESTA ===
1. Inception (2010) - película de ciencia ficción que explora la realidad de una empresa de inversión en personas. 

2. The Dark Knight (2008) - película de superhéroes que explora la realidad del Joker como un villano y el conflicto entre la justicia y la criminalidad.

En cuanto a su contenido, both films exploran complex and dreamlike worlds, and both feature characters with complex motivations and personalities. They also both deal with themes of power, corruption, and the consequences of decision-making.


=== CONTEXTO (Dense) ===
['Inception es una película de ciencia ficción sobre sueños dentro de sueños.', 'The Matrix explora la realidad simulada y la inteligencia artificial.']

=== RESPUESTA ===
1. Inception - una película de ciencia ficción que explora la realidad simulada y la

## 8. Discusión

- Ahora el modelo **sí genera texto real**
- El contexto recuperado influye directamente en la respuesta
- Puedes comparar cómo cambia la respuesta según el tipo de búsqueda